### 0.라이브러리 호출

In [ ]:
# 라이브러리 호출

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
import warnings
import platform

# 전처리
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split, KFold, StratifiedKFold, cross_val_score, GridSearchCV
from sklearn.utils.class_weight import compute_sample_weight # 샘플 가중치: 모델 학습 시 각 샘플에 중요도를 다르게 부여하는 방법

# 모델
from lightgbm import LGBMRegressor                  # LightGBM 회귀모델
from sklearn.ensemble import RandomForestRegressor  # 랜덤 포레스트 회귀모델
from xgboost import XGBRegressor                    # XGBoost 회귀모델

# 모델 학습 및 평가 지표
from sklearn.metrics import (
    mean_squared_error,
    mean_absolute_error, 
    r2_score
)

# SHAP
import shap

# 모델 저장/불러오기
import joblib
import os

# 출력 설정
warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 100)


# 한글 폰트 설정
if platform.system() == 'Windows':
    plt.rcParams['font.family'] = 'Malgun Gothic'
elif platform.system() == 'Darwin':  # macOS
    plt.rcParams['font.family'] = 'AppleGothic'
else:  # Linux
    plt.rcParams['font.family'] = 'NanumGothic'

# matplotlib 설정
plt.rcParams['axes.unicode_minus'] = False  # 마이너스 기호 깨짐 방지
plt.rcParams['figure.figsize'] = (12, 6)

# 시드 고정
np.random.seed(42)

print("=" * 60)
print("라이브러리 로드 완료!")
print("한글 폰트 설정 완료!")
print("=" * 60)

---
### 1. 데이터 로드
- FnB와 IT 롱폼 영상 단위 데이터 통합
- 영상별 긍정 댓글 비율 계산 (`target`)
- 입력 feature 후보:

  **도메인**
  - `domain`

  **영상 특성**
  - `영상길이(초)`, `has_paid_product_placement`, `channel_tier`

  **업로드 시점**
  - `upload_month`, `upload_dayofweek`, `upload_hour`, `upload_quarter`

  **콘텐츠 정보**
  - `description_length`, `tags_count`

  **콘텐츠 분류 결과**
  - `cls_content_type`, `cls_marketing_purpose`, `cls_cta_type`
  - `cls_is_series`, `cls_is_collaboration`

- **성공/실패 여부(`grade`) 제외**
  - `grade`는 조회수, 좋아요 등 영상 성과 기준으로 분류된 변수
  - 타깃 변수인 긍정 댓글 비율과는 별개의 개념
    (성과가 낮아도 긍정 댓글 비율이 높을 수 있고, 반대도 가능)
  - 따라서 `grade`는 입력 변수에서 제외하고
    성공/실패 구분 없이 전체 영상 데이터를 학습에 활용

In [ ]:
# IT ; 성공으로 분류된 롱폼
df_success_IT = pd.read_csv("../../../data/results/main_dataset/롱폼 영상(+콘텐츠 분류 정보)/it_longform_success_data_with_classified.csv", encoding='utf-8')

# IT ; 실패로 분류된 롱폼
df_fail_IT = pd.read_csv("../../../data/results/main_dataset/롱폼 영상(+콘텐츠 분류 정보)/it_longform_fail_data_with_classified.csv", encoding='utf-8')

# FnB ; 성공으로 분류된 롱폼
df_success_FnB = pd.read_csv("../../../data/results/main_dataset/롱폼 영상(+콘텐츠 분류 정보)/fnb_longform_success_data_with_classified.csv", encoding='utf-8')

# FnB ; 실패로 분류된 롱폼
df_fail_FnB = pd.read_csv("../../../data/results/main_dataset/롱폼 영상(+콘텐츠 분류 정보)/fnb_longform_fail_data_with_classified.csv", encoding='utf-8')

In [ ]:
# IT ; 성공으로 분류된 롱폼의 댓글
df_success_IT_comment = pd.read_csv("../../../data/results/main_dataset/댓글(필터링+감성분석 정보)/sentiment_filtered_cleaned_it_longform_success_comment.csv", encoding='utf-8')

# IT ; 실패로 분류된 롱폼의 댓글
df_fail_IT_comment = pd.read_csv("../../../data/results/main_dataset/댓글(필터링+감성분석 정보)/sentiment_filtered_cleaned_it_longform_fail_comment.csv", encoding='utf-8')

# FnB ; 성공으로 분류된 롱폼의 댓글
df_success_FnB_comment = pd.read_csv("../../../data/results/main_dataset/댓글(필터링+감성분석 정보)/sentiment_fnb_longform_success_comment_final.csv", encoding='utf-8')

# FnB ; 실패로 분류된 롱폼의 댓글
df_fail_FnB_comment = pd.read_csv("../../../data/results/main_dataset/댓글(필터링+감성분석 정보)/sentiment_filtered_cleaned_fnb_longform_fail_comment.csv", encoding='utf-8')

In [ ]:
print('[IT]')
print(f"- 성공으로 분류된 롱폼 데이터의 행 개수: {len(df_success_IT)}개")
print(f"- 성공으로 분류된 롱폼 데이터 댓글의 행 개수: {len(df_success_IT_comment)}개")
print(f"- 실패로 분류된 롱폼 데이터의 행 개수: {len(df_fail_IT)}개")
print(f"- 실패로 분류된 롱폼 데이터 댓글의 행 개수: {len(df_fail_IT_comment)}개")

print('\n[FnB]')
print(f"- 성공으로 분류된 롱폼 데이터의 행 개수: {len(df_success_FnB)}개")
print(f"- 성공으로 분류된 롱폼 데이터 댓글의 행 개수: {len(df_success_FnB_comment)}개")
print(f"- 실패로 분류된 롱폼 데이터의 행 개수: {len(df_fail_FnB)}개")
print(f"- 실패로 분류된 롱폼 데이터 댓글의 행 개수: {len(df_fail_FnB_comment)}개")

---
### 2. 전처리

#### 2-1. 데이터 통합
- IT/FnB 성공/실패 롱폼 영상 데이터 통합 → 영상 단위 데이터 1개 (총 1,093개 영상)
- IT/FnB 성공/실패 롱폼 댓글 데이터 통합 → 타깃 변수 계산용 임시 데이터 1개 (총 144,165개 댓글)

#### 2-2. 댓글 필터링
- 이벤트 참여 댓글(`is_event=True`) 제거
  - 이벤트 참여 목적의 댓글은 영상 콘텐츠에 대한 순수한 반응이 아님
- 외국어 댓글(`is_korean=False`) 제거
  - 외국어 댓글은 감성 분류 신뢰도가 낮을 수 있음
  - 분석 대상이 한국 브랜드 채널이므로 한국어 댓글만 사용

#### 2-3. 타깃 변수 생성
- 필터링된 댓글을 `video_id` 기준으로 그룹화
- 영상별 긍정 댓글 비율 계산 → 타깃 변수(`target`) 생성
  - 공식: `긍정 댓글 수 / 전체 댓글 수`
  - 댓글이 없는 영상은 타깃 변수 계산 불가 → 이후 merge 시 제외

#### 2-4. 영상 데이터와 타깃 변수 merge
- 영상 데이터와 타깃 변수를 `video_id` 기준으로 결합
- `how='inner'`로 댓글이 없는 영상은 자동 제외

#### 2-5. feature 선택
- 식별자(`video_id`, `channel_id` 등) 제외
- 성과 지표(`조회수`, `좋아요수`, `score1`, `score2`, `grade` 등) 제외
  - 영상 제작 시점에 알 수 없는 값
  - 타깃 변수와 연관될 수 있어 데이터 누수 위험
- 텍스트 컬럼(`description`, `tags`, `cls_reason` 등) 제외
- 최종 feature 목록 (15개):
  - `domain`, `영상길이(초)`, `has_paid_product_placement`, `channel_tier`
  - `upload_month`, `upload_dayofweek`, `upload_hour`, `upload_quarter`
  - `description_length`, `tags_count`
  - `cls_content_type`, `cls_marketing_purpose`, `cls_cta_type`
  - `cls_is_series`, `cls_is_collaboration`

#### 2-6. 결측값 처리
- 결측값 현황 확인 후 처리 방법 결정
  - 수치형: 중앙값으로 대체
  - 범주형: 최빈값으로 대체 or 별도 범주('unknown')로 처리
  - 결측값이 없다면 별도의 처리는 진행하지 않음

#### 2-7. 범주형 변수 인코딩
- Label Encoding 적용
  - `cls_is_series`, `cls_is_collaboration` (불린 타입)
  - `domain`, `channel_tier`, `cls_content_type`,
    `cls_marketing_purpose`, `cls_cta_type`
- **LightGBM 단일 모델 사용으로 Label Encoding 통일**
  - 트리 기반 모델인 LightGBM은
    Label Encoding된 범주형 변수를 순서가 아닌 분기 기준으로만 사용
  - 범주 간 순서 왜곡 문제 없음

#### 2-8. 데이터 불균형 보정

**타깃 변수 분포 현황**
- U자형 분포 확인
  - 긍정률 0.0 근처: 92개 (긍정 댓글이 거의 없는 영상)
  - 긍정률 1.0 근처: 180개 (긍정 댓글이 대부분인 영상)
  - 중간 구간(0.3~0.7): 상대적으로 적음

**보정 방법**

- **샘플 가중치 (sample_weight)**
  - 각 구간의 샘플 수에 반비례하여 가중치 계산
  - 공식: `가중치 = 전체 샘플 수 / (구간 수 × 해당 구간 샘플 수)`
  - 샘플이 적은 중간 구간에 높은 가중치 부여
  - 모델 학습 시 중간 구간도 균등하게 학습하도록 유도
  - `sklearn`의 `compute_sample_weight`로 자동 계산

- **구간화 기반 KFold**
  - target을 구간으로 나눈 뒤 각 구간이 모든 fold에 고르게 분포되도록 분할
  - 일반 KFold는 특정 fold에 극단값(0.0, 1.0)만 몰릴 수 있어
    검증 결과가 왜곡될 수 있음
  - 구간화 KFold로 더 안정적인 교차검증 가능

In [ ]:
# ========================================
# [전처리 1] 데이터 통합
# ========================================

# [역할] IT/FnB 성공/실패 영상 데이터 및 댓글 데이터 각각 통합
# [근거] 도메인을 입력 변수로 포함하여 통합 모델 학습

# 영상 데이터 통합
df_video = pd.concat([
    df_success_IT,
    df_fail_IT,
    df_success_FnB,
    df_fail_FnB
], ignore_index=True)

# 댓글 데이터 통합 (타깃 변수 계산용)
df_comment = pd.concat([
    df_success_IT_comment,
    df_fail_IT_comment,
    df_success_FnB_comment,
    df_fail_FnB_comment
], ignore_index=True)

print(f"영상 데이터: {len(df_video)}개")
print(f"댓글 데이터: {len(df_comment)}개")

In [ ]:
# ========================================
# [전처리 2] 댓글 필터링
# ========================================

# [역할] 이벤트 참여 댓글 및 외국어 댓글 제거
# [근거] 이벤트 참여 댓글은 영상 콘텐츠에 대한 순수한 반응이 아님
#        외국어 댓글은 감성 분류 신뢰도가 낮을 수 있음

before = len(df_comment)
df_comment = df_comment[df_comment['is_event'] == False]
print(f"이벤트 댓글 제거: {before - len(df_comment)}개 제거 → {len(df_comment)}개 남음")

before = len(df_comment)
df_comment = df_comment[df_comment['is_korean'] == True].reset_index(drop=True)
print(f"외국어 댓글 제거: {before - len(df_comment)}개 제거 → {len(df_comment)}개 남음")

In [ ]:
# ========================================
# [전처리 3] 타깃 변수 생성
# ========================================

# [역할] 영상별 긍정 댓글 비율 계산 → 타깃 변수(target_feature) 생성
# [근거] 전체 댓글 수 대비 긍정 댓글 수의 비율
#        공식: 긍정 댓글 수 / 전체 댓글 수

agg = (
    df_comment.groupby('video_id')['sentiment']
    .agg(
        total='count',
        positive=lambda x: (x == '긍정').sum()
    )
    .reset_index()
)
agg['target_feature'] = agg['positive'] / agg['total']

print(f"타깃 변수 계산 완료: {len(agg)}개 영상")
print(f"\n[긍정 댓글 비율 분포]")
print(agg['target_feature'].describe())

In [ ]:
# 타깃 변수 분포 확인 (시각화)
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# 히스토그램
axes[0].hist(agg['target_feature'], bins=30, color='#4C72B0', alpha=0.85, edgecolor='white')
axes[0].set_title('긍정 댓글 비율 분포', fontsize=13)
axes[0].set_xlabel('긍정 댓글 비율')
axes[0].set_ylabel('영상 수')
axes[0].axvline(agg['target_feature'].mean(),   color='red',   linestyle='--', label=f'평균: {agg["target_feature"].mean():.2f}')
axes[0].axvline(agg['target_feature'].median(), color='green', linestyle='--', label=f'중앙값: {agg["target_feature"].median():.2f}')
axes[0].legend()

# 박스플롯
axes[1].boxplot(agg['target_feature'], vert=True)
axes[1].set_title('긍정 댓글 비율 박스플롯', fontsize=13)
axes[1].set_ylabel('긍정 댓글 비율')

plt.tight_layout()
plt.show()

In [ ]:
# ========================================
# [전처리 4] 영상 데이터와 타깃 변수 merge
# ========================================

# [역할] 영상 특성 데이터와 타깃 변수를 결합
# [작업] how='inner'로 댓글이 없는 영상은 제외

df = pd.merge(df_video, agg[['video_id', 'target_feature']], on='video_id', how='inner')

print(f"merge 후 데이터: {len(df)}개")
print(f"\n[NaN 현황]")
print(df.isna().sum())

In [ ]:
# ========================================
# [전처리 5] feature 선택
# ========================================

# [역할] 모델 학습에 사용할 feature만 추출
# [근거] 식별자, 성과 지표, 텍스트 컬럼 제외

FEATURES = [
    # 도메인
    'domain',

    # 영상 특성
    '영상길이(초)',
    'has_paid_product_placement',
    'channel_tier',

    # 업로드 시점
    'upload_month',
    'upload_dayofweek',
    'upload_hour',
    'upload_quarter',

    # 콘텐츠 정보
    'description_length',
    'tags_count',

    # 콘텐츠 분류 결과
    'cls_content_type',
    'cls_marketing_purpose',
    'cls_cta_type',
    'cls_is_series',
    'cls_is_collaboration',
]

TARGET = 'target_feature' # 영상별 긍정 댓글 비율

df_model = df[FEATURES + [TARGET]].copy()

print(f"최종 데이터: {len(df_model)}행 / {len(FEATURES)}개 feature")
print(f"\n[NaN 현황]")
print(df_model.isna().sum())
print(f"\n[feature 데이터 타입]")
print(df_model.dtypes)

In [ ]:
# ========================================
# [전처리 6] 범주형 변수 인코딩
# ========================================

# [역할] 문자열/불린 타입 컬럼을 숫자형으로 변환
# [근거] 트리 기반 모델(랜덤 포레스트, LightGBM)은 Label Encoding된 범주형 변수를
#      순서가 아닌 분기의 기준으로만 사용하므로 문제없음

# Label Encoding이란?
# 문자열 범주형 변수를 숫자로 변환하는 방법
# 예) domain: 'FnB' → 0, 
#             'IT' → 1
#    cls_content_type: '광고/CF' → 0, 
#                      '기술설명' → 1, 
#                      '브이로그' → 2 ...
# 알파벳/가나다 순으로 숫자를 부여하며, 트리 기반 모델은 숫자의 크고 작음이 아닌 같은지 다른지만 보므로
# 순서가 없는 범주형 변수에도 문제없이 적용 가능하다고 함

# str 타입 컬럼 Label Encoding
str_cols = [
    'domain', 'channel_tier', 'upload_dayofweek',
    'cls_content_type', 'cls_marketing_purpose', 'cls_cta_type'
]

le = LabelEncoder()
for col in str_cols:
    df_model[col] = le.fit_transform(df_model[col])
    print(f"{col} 인코딩 완료: {df_model[col].unique()}")

# bool 타입 컬럼 int 변환
# True → 1, False → 0으로 변환
bool_cols = ['has_paid_product_placement', 'cls_is_series', 'cls_is_collaboration']
for col in bool_cols:
    df_model[col] = df_model[col].astype(int)
    print(f"{col} 변환 완료: {df_model[col].unique()}")

print(f"\n[인코딩 후 데이터 타입]")
print(df_model.dtypes)

In [ ]:
# ========================================
# [전처리 6-1] 인코딩 매핑 정보 저장
# ========================================

# [역할] 각 범주형 변수의 인코딩 매핑 정보를 딕셔너리로 저장
# [근거] SHAP 결과 해석 시 숫자 → 원래 범주명으로 역변환 필요

encoding_map = {}

le = LabelEncoder()
for col in str_cols:
    # 원본 데이터에서 fit하여 매핑 정보 추출
    le.fit(df[col])
    encoding_map[col] = {i: cls for i, cls in enumerate(le.classes_)}
    print(f"\n[{col}]")
    for num, label in encoding_map[col].items():
        print(f"  {num} → {label}")

In [ ]:
# ========================================
# [전처리 7] 데이터 불균형 보정
# ========================================

# [역할] 타깃 변수의 U자형 분포에 대한 보정
# [근거] 긍정률 0.0 근처와 1.0 근처에 데이터가 몰려있어
#        중간 구간이 학습에서 소외될 수 있음

# target을 5개 구간으로 나누기
# [작업] pd.cut으로 target_feature를 구간으로 분할
#        구간: 0.0~0.2, 0.2~0.4, 0.4~0.6, 0.6~0.8, 0.8~1.0

# labels: 각 구간에 숫자를 붙이는 이름
#         0.0~0.2 → 0, 0.2~0.4 → 1, 0.4~0.6 → 2, 0.6~0.8 → 3, 0.8~1.0 → 4
#         compute_sample_weight가 숫자형 입력만 받으므로 숫자로 변환

bins = [0.0, 0.2, 0.4, 0.6, 0.8, 1.01]
labels = [0, 1, 2, 3, 4]

target_bin = pd.cut(
    df_model[TARGET],
    bins=bins,
    labels=labels,
    include_lowest=True
)

# 구간별 샘플 수 확인
print("[구간별 샘플 수]")
print(target_bin.value_counts().sort_index())

# 샘플 가중치 계산
# [공식] 가중치 = 전체 샘플 수 / (구간 수 × 해당 구간 샘플 수)
# [효과] 샘플이 적은 중간 구간에 높은 가중치 부여
#        → 모델이 중간 구간도 균등하게 학습하도록 유도
sample_weights = compute_sample_weight(class_weight='balanced', y=target_bin)

print(f"\n[샘플 가중치 요약]")
print(f"- 최솟값: {sample_weights.min():.4f}")
print(f"- 최댓값: {sample_weights.max():.4f}")
print(f"- 평균  : {sample_weights.mean():.4f}")

# 구간별 가중치 확인
weight_df = pd.DataFrame({
    'target_feature': df_model[TARGET],
    'target_bin': target_bin,
    'sample_weight': sample_weights
})

print("\n[구간별 가중치]")
display(
    weight_df.groupby('target_bin')['sample_weight']
    .agg(샘플수='count', 가중치='first')
    .reset_index()
)

---
### 3. 데이터 분할
- Train / Test = 8 : 2
- `random_state = 42`로 고정
- `stratify=target_bin` 적용
  - 타깃 변수를 5개 구간으로 나눈 `target_bin` 기준으로 층화 분할
  - 각 구간의 비율이 train/test에 고르게 분포되도록 보장
  - 데이터 불균형(U자형 분포)으로 인해 특정 구간이 한쪽에 몰리는 문제 방지

In [ ]:
# ========================================
# 데이터 분할
# ========================================

# [역할] 학습 데이터와 테스트 데이터 분리
# [근거] 모델 학습에 사용하지 않은 데이터로 성능을 평가해야
#        실제 예측 성능을 신뢰할 수 있음

# feature 행렬과 타깃 벡터 분리
X = df_model[FEATURES].values
y = df_model[TARGET].values

# Train / Test = 8 : 2
# stratify=target_bin: 구간 비율이 train/test에 고르게 분포되도록 분할
# np.arange(len(df_model)): train 인덱스를 직접 받아와서
#                           target_bin_train 추출 시 데이터 누수 방지
X_train, X_test, y_train, y_test, weights_train, weights_test, idx_train, idx_test = train_test_split(
    X, y, sample_weights, np.arange(len(df_model)),
    test_size=0.2,
    random_state=42,
    stratify=target_bin
)

# train 인덱스 기반으로 target_bin_train 추출
# [근거] train_test_split은 랜덤으로 섞어서 나누므로
#        앞쪽 N개가 반드시 train에 해당한다는 보장이 없음
#        → 인덱스를 직접 받아와서 정확하게 추출해야 데이터 누수 방지
target_bin_train = target_bin.iloc[idx_train].reset_index(drop=True)

print(f"전체 데이터  : {len(X)}개")
print(f"학습 데이터  : {len(X_train)}개")
print(f"테스트 데이터: {len(X_test)}개")

In [ ]:
# ========================================
# [데이터 저장] 학습/테스트 데이터셋 저장
# ========================================

# [역할] 학습/테스트 데이터셋을 파일로 저장
# [근거] 모델 재학습 시 동일한 데이터셋을 사용하기 위함
#      재현 가능성 보장

# 저장 폴더 생성
save_dir = "../../../works/Haeun/analysis_stats+ML/modeling/longform"
os.makedirs(save_dir, exist_ok=True)

# 학습 데이터 저장
pd.DataFrame(X_train, columns=FEATURES).assign(target_feature=y_train).to_csv(
    f"{save_dir}/train_data(longform).csv", index=False, encoding='utf-8'
)

# 테스트 데이터 저장
pd.DataFrame(X_test, columns=FEATURES).assign(target_feature=y_test).to_csv(
    f"{save_dir}/test_data(longform).csv", index=False, encoding='utf-8'
)

# 샘플 가중치 저장
pd.DataFrame({'sample_weight': weights_train}).to_csv(
    f"{save_dir}/train_weights(longform).csv", index=False, encoding='utf-8'
)

print(f"저장 완료: {save_dir}")
print(f"- train_data(longform).csv : {len(X_train)}개")
print(f"- test_data(longform).csv : {len(X_test)}개")
print(f"- train_weights(longform).csv : {len(weights_train)}개")

---
### 4. 모델 학습 및 평가

#### 4-1. 기본 모델(default) 성능 비교
- LightGBM, 랜덤 포레스트, XGBoost 각각 default 파라미터로 학습
- 구간화 기반 5-Fold CV로 성능 비교 (RMSE, MAE, R²)
- 세 모델 성능 비교 후 튜닝 방향 결정

In [ ]:
# 기본 모델(default) 성능 비교
# [역할] LightGBM, 랜덤 포레스트, XGBoost 각각 default 파라미터로 학습
# [근거] 튜닝 전 기본 성능을 비교하여 튜닝 방향 결정

# [작업] 비교할 모델 3가지 정의
#      - 동일한 random_state로 재현 가능성 보장
models = {
    'LightGBM' : LGBMRegressor(random_state=42),
    'RandomForest': RandomForestRegressor(random_state=42),
    'XGBoost' : XGBRegressor(random_state=42)
}

# 구간화 기반 5-Fold Cross Validation
# - n_splits=5: 5등분
# - shuffle=True: 나누기 전에 섞기 (순서 편향 방지를 위해)
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

# 각 모델별 CV 결과 저장
default_results = {}

for model_name, model in models.items():
    # 각 모델의 Fold별 결과가 담길 리스트
    cv_rmse = []
    cv_mae  = []
    cv_r2 = []

    # 5-Fold CV 구현
    # - skf.split(X_train, target_bin_train): 구간 비율을 유지하며 train 인덱스/val 인덱스를 5번 나눔
    # - tr_idx: 이번 fold에서 학습에 쓸 인덱스
    # - val_idx: 이번 fold에서 검증에 쓸 인덱스
    # - X_tr, y_tr : 이번 fold 학습 데이터
    # - X_val, y_val : 이번 fold의 중간 점검 데이터
    for tr_idx, val_idx in skf.split(X_train, target_bin_train):
        X_tr, X_val = X_train[tr_idx], X_train[val_idx]
        y_tr, y_val = y_train[tr_idx], y_train[val_idx]
        w_tr = weights_train[tr_idx]

        # sample_weight=w_tr: 중간 구간에 높은 가중치를 줘서 균등하게 학습시킴
        model.fit(X_tr, y_tr, sample_weight=w_tr)

        y_pred_val = model.predict(X_val)

        cv_rmse.append(np.sqrt(mean_squared_error(y_val, y_pred_val))) # 오차를 제곱해서 평균낸 뒤 루트 (큰 오차에 민감한 편)
        cv_mae.append(mean_absolute_error(y_val, y_pred_val))          # 오차의 절댓값 평균
        cv_r2.append(r2_score(y_val, y_pred_val))                      # 1에 가까울수록 좋음

    default_results[model_name] = {
        'RMSE': np.mean(cv_rmse),
        'MAE' : np.mean(cv_mae),
        'R²' : np.mean(cv_r2)
    }

    print(f"\n[{model_name}]")
    print(f"- RMSE: {np.mean(cv_rmse):.4f} (±{np.std(cv_rmse):.4f})")
    print(f"- MAE : {np.mean(cv_mae):.4f} (±{np.std(cv_mae):.4f})")
    print(f"- R²  : {np.mean(cv_r2):.4f} (±{np.std(cv_r2):.4f})")

# 성능 비교 요약
print(f"\n{'='*50}")
print(f"[기본 모델 성능 비교 요약]")
print(f"{'모델':<15} {'RMSE':>8} {'MAE':>8} {'R²':>8}")
print(f"{'-'*40}")
for model_name, result in default_results.items():
    print(f"{model_name:<15} {result['RMSE']:>8.4f} {result['MAE']:>8.4f} {result['R²']:>8.4f}")

#### **📊결과 해석**
- 랜덤포레스트
    - RMSE가 가장 낮고, R²이 가장 높음 -> default 모델을 기준으로 가장 성능이 좋았음
    - 단, R²의 표준편차가 0.082로 꽤 큰 편이라 fold마다 성능이 들쭉날쭉했던 것으로 보임

- LightGBM
    - R²의 표준편차가 0.0278로 세 모델 중 가장 작은 편 -> 가장 안정적인 성능
    - RMSE는 중간 정도이지만, 표준편차는 가장 작음 (±0.0075)

- XGBoost
    - RMSE가 가장 크고, R²은 가장 낮음 -> default 모델을 기준으로 가장 성능이 낮음
    - 표준편차도 세 모델 중 가장 큼

---
### 5. 하이퍼파라미터 튜닝
- RMSE를 성능 개선의 기준으로 선택한 이유
    - 큰 오차에 더 민감하게 반응함
    - 긍정 댓글 비율 예측에서 실제값 0.9를 0.3으로 예측하는 것과 같은 극단적인 오류에 강한 패널티를 부여하기 위해

- MAE를 쓰지 않은 이유
    - 모든 오차를 동등하게 처리해 샘플 수가 적은 중간 구간(0.2~0.8)의 잘못된 예측이 전체 지표에 거의 반영되지 않기 때문에

- R²를 쓰지 않은 이유
    - 데이터 분포에 따라 왜곡될 수 있기 때문에

#### 5-1. LightGBM 튜닝
- GridSearchCV + 구간화 기반 5-Fold CV

In [ ]:
# ========================================
# [5-1] LightGBM 하이퍼파라미터 튜닝
# ========================================

# [역할] LightGBM 하이퍼파라미터 최적화
# [근거] 기본 모델 성능(RMSE 0.2954) 개선을 위해 GridSearchCV로 최적 파라미터 탐색

param_grid_lgb = {
    'num_leaves':        [10, 15, 20, 25],      # 트리의 최대 리프 수 -> 크게 할수록 세밀하게 학습 가능하지만 과적합 위험
    'min_child_samples': [15, 20, 25, 30],      # 리프 노드의 최소 샘플 수 (과적합 방지)
    'learning_rate':     [0.005, 0.01, 0.02],  # 학습률: 작을수록 천천히 학습
    'n_estimators':      [150, 200, 300, 500], # 트리 개수
    'max_depth':         [3, 5, 7],             # 트리 최대 깊이
}

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

grid_lgb = GridSearchCV(
    estimator=LGBMRegressor(random_state=42),
    param_grid=param_grid_lgb,
    cv=skf.split(X_train, target_bin_train),
    scoring='neg_root_mean_squared_error',      # 최적 파라미터 선택 기준: neg_RMSE (=RMSE를 음수로 만듦)
    n_jobs=-1,                                  # 모든 CPU 코어 사용
    verbose=1                                   # 학습 진행 상황 출력 옵션 (1: 간략하게)
)

grid_lgb.fit(X_train, y_train, sample_weight=weights_train)

print(f"\n[LightGBM 최적 하이퍼파라미터]")
print(grid_lgb.best_params_)
print(f"\n[LightGBM 최적 CV RMSE]")
print(f"{-grid_lgb.best_score_:.4f}")

#### 5-2. 랜덤 포레스트 튜닝
- GridSearchCV + 구간화 기반 5-Fold CV

In [ ]:
# ========================================
# [5-2] 랜덤 포레스트 하이퍼파라미터 튜닝
# ========================================

# [역할] 랜덤 포레스트 하이퍼파라미터 최적화
# [근거] 기본 모델 성능(RMSE 0.2806) 개선을 위해 GridSearchCV로 최적 파라미터 탐색

param_grid_rf = {
    'n_estimators':      [100, 200, 300, 500],  # 트리 개수: 많을수록 안정적이지만 속도 느림
    'max_depth':         [None, 5, 10, 20],      # 트리 최대 깊이 (None: 제한 없음)
    'min_samples_split': [2, 5, 10],             # 노드 분할을 위한 최소 샘플 수
    'min_samples_leaf':  [1, 2, 4],              # 리프 노드의 최소 샘플 수 (과적합 방지)
    'max_features':      ['sqrt', 'log2'],       # 분기 시 사용할 feature 수
                                                  # sqrt: 전체 feature 수의 제곱근(기본값)
                                                  # log2: 전체 feature 수의 log2
}

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

grid_rf = GridSearchCV(
    estimator=RandomForestRegressor(random_state=42),
    param_grid=param_grid_rf,
    cv=skf.split(X_train, target_bin_train),
    scoring='neg_root_mean_squared_error',      # 최적 파라미터 선택 기준: neg_RMSE (=RMSE를 음수로 만듦)
    n_jobs=-1,                                  # 모든 CPU 코어 사용
    verbose=1                                   # 학습 진행 상황 출력 옵션 (1: 간략하게)
)

grid_rf.fit(X_train, y_train, sample_weight=weights_train)

print(f"\n[랜덤 포레스트 최적 하이퍼파라미터]")
print(grid_rf.best_params_)
print(f"\n[랜덤 포레스트 최적 CV RMSE]")
print(f"{-grid_rf.best_score_:.4f}")

#### 5-3. XGBoost 튜닝
- GridSearchCV + 구간화 기반 5-Fold CV

In [ ]:
# ========================================
# [5-3] XGBoost 하이퍼파라미터 튜닝
# ========================================

# [역할] XGBoost 하이퍼파라미터 최적화
# [근거] 기본 모델 성능(RMSE 0.3067) 개선을 위해 GridSearchCV로 최적 파라미터 탐색

param_grid_xgb = {
    'n_estimators':  [100, 200, 300, 500],  # 트리 개수
    'max_depth':     [3, 5, 7, 10],         # 트리 최대 깊이
    'learning_rate': [0.005, 0.01, 0.05, 0.1],  # 학습률: 작을수록 천천히 학습
    'subsample':     [0.6, 0.8, 1.0],       # 각 트리 학습 시 사용할 샘플 비율
                                             # 1.0 미만이면 랜덤 샘플링 → 과적합 방지
    'colsample_bytree': [0.6, 0.8, 1.0],   # 각 트리 학습 시 사용할 feature 비율
                                             # 1.0 미만이면 랜덤 feature 선택 → 과적합 방지
}

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

grid_xgb = GridSearchCV(
    estimator=XGBRegressor(random_state=42),
    param_grid=param_grid_xgb,
    cv=skf.split(X_train, target_bin_train),
    scoring='neg_root_mean_squared_error',  # 최적 파라미터 선택 기준: neg_RMSE
    n_jobs=-1,                               # 모든 CPU 코어 사용
    verbose=1                                # 학습 진행 상황 출력 옵션 (1: 간략하게)
)

grid_xgb.fit(X_train, y_train, sample_weight=weights_train)

print(f"\n[XGBoost 최적 하이퍼파라미터]")
print(grid_xgb.best_params_)
print(f"\n[XGBoost 최적 CV RMSE]")
print(f"{-grid_xgb.best_score_:.4f}")

In [ ]:
# ========================================
# [5-4] 튜닝 후 모델 성능 비교 요약
# ========================================

# [역할] 세 모델의 튜닝 후 성능을 비교하여 최종 모델 선택 근거 마련

# best_score_는 neg_RMSE(음수)로 저장되기 때문에 -를 붙여서 양수 RMSE로 변환해 출력함
tuned_results = {
    'LightGBM'    : {'RMSE': -grid_lgb.best_score_},
    'RandomForest': {'RMSE': -grid_rf.best_score_},
    'XGBoost'     : {'RMSE': -grid_xgb.best_score_}
}

print(f"[튜닝 후 모델 성능 비교 요약]")
print(f"{'모델':<15} {'기본 RMSE':>12} {'튜닝 후 RMSE':>14} {'개선폭':>8}")
print(f"{'-'*52}")
print(f"{'LightGBM':<15} {0.2954:>12.4f} {-grid_lgb.best_score_:>14.4f} {0.2954 - (-grid_lgb.best_score_):>8.4f}")
print(f"{'RandomForest':<15} {0.2806:>12.4f} {-grid_rf.best_score_:>14.4f} {0.2806 - (-grid_rf.best_score_):>8.4f}")
print(f"{'XGBoost':<15} {0.3067:>12.4f} {-grid_xgb.best_score_:>14.4f} {0.3067 - (-grid_xgb.best_score_):>8.4f}")

#### 5-4. 최종 모델 선택
- 세 가지 모델의 하이퍼파라미터 튜닝 후, CV 성능 비교 결과

  | 모델 | 기본 RMSE | 튜닝 후 RMSE | 개선폭 |
  |---|---|---|---|
  | LightGBM | 0.2954 | 0.2728 | 0.0226 |
  | RandomForest | 0.2806 | 0.2718 | 0.0088 |
  | XGBoost | 0.3067 | 0.2716 | 0.0351 |

- 하이퍼파라미터 튜닝을 거친 XGBoost가 RMSE 0.2716으로 가장 좋은 성능을 보임
  - 단, RandomForest(0.2718)와 차이가 0.0002로 거의 동일함

-> test 데이터로 최종 성능 평가 후 최종 모델 확정 예정

In [ ]:
# ========================================
# [5-4] 최종 모델 선택 - Test 데이터 평가
# ========================================

# [역할] 튜닝된 세 모델을 전체 train 데이터로 재학습 후
#        test 데이터로 최종 성능 평가
# [근거] CV 성능은 train 데이터 내에서의 검증이므로
#        실제 예측 성능은 test 데이터로 최종 확인해야 함

# 튜닝된 최적 파라미터로 모델 생성
# ** : 딕셔너리를 언패킹하여 파라미터로 전달
# 예) LGBMRegressor(**grid_lgb.best_params_)
#   = LGBMRegressor(learning_rate=0.005, max_depth=7, ...)
# 직접 하나씩 입력하지 않아도 되므로 코드가 간결하고 실수 방지
final_models = {
    'LightGBM' : LGBMRegressor(**grid_lgb.best_params_, random_state=42),
    'RandomForest': RandomForestRegressor(**grid_rf.best_params_, random_state=42),
    'XGBoost' : XGBRegressor(**grid_xgb.best_params_, random_state=42)
}

final_results = {}

for model_name, model in final_models.items():
    # 전체 train 데이터로 재학습
    model.fit(X_train, y_train, sample_weight=weights_train)

    # test 데이터로 예측
    y_pred_test = model.predict(X_test)

    # 성능 평가
    rmse = np.sqrt(mean_squared_error(y_test, y_pred_test))
    mae  = mean_absolute_error(y_test, y_pred_test)
    r2   = r2_score(y_test, y_pred_test)

    final_results[model_name] = {
        'RMSE': rmse,
        'MAE' : mae,
        'R²' : r2
    }

    print(f"\n[{model_name}]")
    print(f"- RMSE: {rmse:.4f}")
    print(f"- MAE : {mae:.4f}")
    print(f"- R²  : {r2:.4f}")

# 최종 성능 비교 요약
print(f"\n{'='*50}")
print(f"[최종 모델 성능 비교 (Test 데이터)]")
print(f"{'모델':<15} {'RMSE':>8} {'MAE':>8} {'R²':>8}")
print(f"{'-'*40}")
for model_name, result in final_results.items():
    print(f"{model_name:<15} {result['RMSE']:>8.4f} {result['MAE']:>8.4f} {result['R²']:>8.4f}")

---
#### **📊튜닝 후 CV 성능 비교 결과 해석**
- XGBoost
    - 하이퍼파라미터 튜닝 후 RMSE 0.2716으로 세 모델 중 가장 낮음
    - 개선폭도 0.0351로 가장 크게 개선됨
    - default에서 가장 낮은 성능이었으나 하이퍼파라미터 튜닝 후 역전

- 랜덤 포레스트
    - 하이퍼파라미터 튜닝 후 RMSE 0.2718로 XGBoost와 차이가 0.0002로 거의 동일
    - 개선폭은 0.0088로 세 모델 중 가장 작음
    - default에서 이미 성능이 좋았기 때문에 개선의 여지가 적었던 것으로 보임

- LightGBM
    - 하이퍼파라미터 튜닝 후 RMSE 0.2728로 세 모델 중 가장 높음
    - 개선폭은 0.0226으로 중간

---
#### **📊Test 데이터 최종 성능 비교 결과 해석**
- XGBoost
    - RMSE 0.2925, MAE 0.2483, R² 0.3385로 세 가지 지표 모두에서 가장 좋은 성능
    - Train 데이터에서의 RMSE(0.2716) 대비 Test 데이터에서의 RMSE(0.2925) 하락폭이 세 모델 중 가장 작음
    - → 상대적으로 일반화 성능이 가장 좋았던 것으로 보임 
    → **최종 모델로 선택**

- LightGBM / 랜덤 포레스트
    - Train 데이터에서의 성능 대비 Test 데이터에서의 성능 하락폭이 XGBoost보다 큼
    - → 상대적으로 일반화 성능이 낮음

---
### 6. 최종 선정된 모델 저장
- 최종 선택된 XGBoost 모델 저장
    - 파일명: longform_model.pkl

In [ ]:
# ===============
# 모델 저장
# ===============
final_model = final_models['XGBoost']

joblib.dump(final_model, f"{save_dir}/longform_model.pkl")
print(f"모델 저장 완료: {save_dir}/longform_model.pkl")

---
### 7. 변수 영향력 분석 (SHAP - 분석용)
- 최종 모델(XGBoost)에 SHAP 적용 (학습 데이터 전체 대상)
- 목적: "어떤 변수가 전반적으로 긍정 댓글 비율에 영향을 주는가?" 파악

- **Summary Plot**
  - 전체 변수의 중요도와 영향 방향을 한눈에 확인 가능
  - 가로축: SHAP 값 (양수: 긍정 비율 높임 / 음수: 긍정 비율 낮춤)
  - 색상: 빨강(변수값 높음) / 파랑(변수값 낮음)

- **Bar Plot**
  - 변수 중요도 순위 확인 가능
  - SHAP 값의 절댓값 평균으로 중요도 측정

- **Dependence Plot**
  - 특정 변수값에 따라 예측이 어떻게 달라지는지 확인 가능
  - 중요도가 높은 변수 위주로 적용됨

In [ ]:
# ========================================
# [7] 변수 영향력 분석 (SHAP - 분석용)
# ========================================

# [역할] 최종 모델(XGBoost)에 SHAP 적용하여 어떤 변수가 긍정 댓글 비율에 영향을 주는지 분석
# SHAP은 각 변수가 예측값에 얼마나, 어떤 방향으로 영향을 줬는지 설명해주는 도구

# SHAP explainer 생성
explainer = shap.Explainer(final_model, X_train)

# SHAP 값 계산 (학습 데이터 전체 대상)
shap_values = explainer(X_train)

# ── Summary Plot ──────────────────────────────────────────
# [역할] 전체 변수의 중요도 + 영향 방향을 한눈에 확인
# [해석] 가로축: SHAP 값 (양수: 긍정 비율 높임, 음수: 긍정 비율 낮춤)
#        색상: 빨강(변수값 높음), 파랑(변수값 낮음)
plt.figure()
shap.summary_plot(shap_values, X_train, feature_names=FEATURES)
plt.tight_layout()
plt.show()

# ── Bar Plot ──────────────────────────────────────────────
# [역할] 변수 중요도 순위 확인
# [해석] SHAP 값의 절댓값 평균으로 중요도 측정
plt.figure()
shap.summary_plot(shap_values, X_train, feature_names=FEATURES, plot_type='bar')
plt.tight_layout()
plt.show()

In [ ]:
# Dependence Plot
# [역할] 특정 변수값에 따라 예측이 어떻게 달라지는지 확인
# [작업] 중요도 상위 5개 변수 위주로 적용

top_features = [
    'cls_is_series',
    'cls_marketing_purpose',
    'cls_cta_type',
    'tags_count',
    'domain'
]

for feature in top_features:
    feat_idx = FEATURES.index(feature)
    fig, ax = plt.subplots(figsize=(10, 6))
    shap.dependence_plot(
        feat_idx,
        shap_values.values,
        X_train,
        feature_names=FEATURES,
        ax=ax
    )
    ax.set_title(f'Dependence Plot: {feature}', fontsize=13)
    plt.tight_layout()
    plt.show()

#### 📊 mapping된 범주
아래와 같이 mapping된 값을 기준으로 그래프 해석 가능

- **domain**: 0=FnB / 1=IT

- **cls_marketing_purpose**:
  - 0=고객유입 / 1=고객유지 / 2=기업이미지 / 3=브랜드캠페인
  - 4=사회공헌/환경 / 5=서비스활용 / 6=정보제공 / 7=제품홍보 / 8=채용

- **cls_cta_type**:
  - 0=구독유도 / 1=구매유도 / 2=기타 / 3=방문유도
  - 4=앱다운로드 / 5=이벤트참여 / 6=정보탐색

- **cls_content_type**:
  - 0=광고/CF / 1=기술설명 / 2=기타 / 3=브이로그 / 4=시설소개
  - 5=애니메이션 / 6=에피소드소개 / 7=영양정보 / 8=요리/레시피
  - 9=웹드라마 / 10=웹예능 / 11=이벤트/행사 / 12=인터뷰
  - 13=제품리뷰 / 14=튜토리얼

- **cls_is_series**: 0=단독 / 1=시리즈

- **cls_is_collaboration**: 0=단독 / 1=콜라보

- **channel_tier**:
  - 0=macro / 1=mega / 2=micro / 3=mid / 4=nano / 5=pico

- **upload_dayofweek**:
  - 0=금요일 / 1=목요일 / 2=수요일 / 3=월요일
  - 4=일요일 / 5=토요일 / 6=화요일

---
#### 📊 변수별 영향 방향 (Dependence Plot)

- **cls_is_series (시리즈 여부)**
  - 시리즈물(1): SHAP 양수 → 긍정적인 댓글의 비율 높임
  - 단독(0): SHAP 음수 → 긍정적인 댓글의 비율 낮춤

- **cls_marketing_purpose (마케팅 목적)**
  - 고객유입(0), 고객유지(1), 기업이미지(2), 브랜드캠페인(3): SHAP 양수 → 긍정 비율 높임
  - 서비스활용(5), 정보제공(6): SHAP 음수 → 긍정 비율 낮춤

- **cls_cta_type (CTA 유형)**
  - 이벤트참여(5): SHAP 가장 높음 → 긍정 비율 가장 높임
  - 정보탐색(6): SHAP 음수 → 긍정 비율 낮춤

- **tags_count (태그 수)**
  - 0~10개: SHAP 양수 → 긍정 비율 높임
  - 20개 이상: SHAP 음수 → 긍정 비율 낮춤

- **domain (도메인)**
  - FnB(0): SHAP 양수 → 긍정 비율 높임
  - IT(1): SHAP 음수 → 긍정 비율 낮춤
  - FnB 도메인이 IT보다 긍정적인 댓글의 비율이 높은 경향 o

---
### 모델 활용 방향

**1. 변수 영향력 분석 (메인)**
- SHAP을 활용하여 긍정 댓글 비율에 영향을 주는 변수와 그 방향을 파악하는 것에 집중
- "어떤 특성의 영상이 긍정 반응을 이끌어내는가"에 대한 인사이트 도출 → 영상 제작 가이드로 활용

**2. 예측 (보조)**
- 절대적인 예측값보다 상대적 비교로 활용
  → "이 조합은 다른 영상보다 긍정 비율이 높을 가능성이 있습니다"
- 모델 성능(RMSE: 0.2925, R²: 0.3385)의 한계를 명시하여 예측값을 참고용으로만 제공

**3. 성능 한계 원인**
- 데이터 수 부족 (606개)
- 긍정 댓글 비율은 영상 특성 외에도
  시사성, 트렌드, 댓글 문화 등
  모델이 학습할 수 없는 요인에 영향을 받음

In [ ]:
# ========================================
# [8] 예측 함수 작성
# ========================================

# [역할] 페르소나가 영상 특성을 입력하면
#       예측한 긍정적인 댓글의 비율과 SHAP 기반 영향 변수를 반환
# - 절대적인 예측값보다 상대적 비교와 영향 변수 중심으로 활용하면 좋을 것 같음

def predict_positive_ratio(input_data: dict) -> None:
    """
    Parameters
    ----------
    input_data : 영상 특성 딕셔너리
        예) {
            'domain': 0,                      # 0=FnB, 1=IT
            '영상길이(초)': 600,
            'has_paid_product_placement': 0,  # 0=없음, 1=있음
            'channel_tier': 2,                # 0=macro~5=pico
            'upload_month': 3,
            'upload_dayofweek': 3,            # 0=금~6=화
            'upload_hour': 18,
            'upload_quarter': 1,
            'description_length': 200,
            'tags_count': 5,
            'cls_content_type': 10,           # 0=광고/CF~14=튜토리얼
            'cls_marketing_purpose': 2,       # 0=고객유입~8=채용
            'cls_cta_type': 5,                # 0=구독유도~6=정보탐색
            'cls_is_series': 1,               # 0=단독, 1=시리즈
            'cls_is_collaboration': 0         # 0=단독, 1=콜라보
        }
    """

    # 입력 데이터 변환
    # [작업] 딕셔너리 → numpy array (모델 입력 형식)
    X_input = np.array([[input_data[f] for f in FEATURES]])

    # 예측 
    pred = final_model.predict(X_input)[0]
    pred = np.clip(pred, 0, 1)  # 0~1 범위로 클리핑

    # 전체 train 데이터와 상대적 비교
    # [작업] 예측값이 train 데이터 전체 예측값 중 상위 몇 %인지 계산
    all_preds = final_model.predict(X_train)
    percentile = (all_preds < pred).mean() * 100

    print(f"{'='*50}")
    print(f"[예측 결과]")
    print(f"  예측 긍정 댓글 비율: {pred:.4f} ({pred*100:.1f}%)")
    print(f"  → 학습 데이터 내 영상 중 상위 {100-percentile:.1f}%에 해당")
    print(f"  ※ 모델 성능 한계(RMSE=0.2925)로 인해 참고용으로만 활용 권장")

    # SHAP 기반 영향 변수 탐색
    # [작업] 입력 데이터 1개에 대한 SHAP 값 계산
    #        어떤 변수가 이 예측값에 얼마나 기여했는지 확인
    shap_input = explainer(X_input)
    shap_vals  = shap_input.values[0]

    # SHAP 값 절댓값 기준 상위 5개 변수 추출
    top_idx = np.argsort(np.abs(shap_vals))[::-1][:5]

    print(f"\n[이 예측값에 가장 큰 영향을 준 변수 Top 5]")
    for idx in top_idx:
        direction = '↑ 긍정 비율 높임' if shap_vals[idx] > 0 else '↓ 긍정 비율 낮춤'
        print(f"  {FEATURES[idx]:<30}: {shap_vals[idx]:+.4f}  ({direction})")
    print(f"{'='*50}")

In [ ]:
# 테스트 실행
sample_input = {
    'domain': 0,
    '영상길이(초)': 600,
    'has_paid_product_placement': 0,
    'channel_tier': 2,
    'upload_month': 3,
    'upload_dayofweek': 3,
    'upload_hour': 18,
    'upload_quarter': 1,
    'description_length': 200,
    'tags_count': 5,
    'cls_content_type': 10,
    'cls_marketing_purpose': 2,
    'cls_cta_type': 5,
    'cls_is_series': 1,
    'cls_is_collaboration': 0
}

predict_positive_ratio(sample_input)